Volatility Features:

- Volatility is a proxy for economic uncertainty. Premium for safe assets ( high quality) may incease while cyclical factors (value) may suffer.
- Local volatility provides a good signal for risk-on/risk-off environments.
- Global volatility spills over across international markets.


In [36]:
# run script from the data_input file
import os
os.chdir('C:/Users/p528552/OneDrive - South African Reserve Bank/Documents/1. MEng - Data Science/1. Project_2025/Data/factor_timing/data_input')

In [37]:
# Libraries:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA


In [38]:
# Import data:
df = pd.read_csv("local_global_vol.csv", parse_dates=['Date'])
df = df.set_index('Date')
df.head()

,SPX,JALSH,VIX,MOVE,SAVIT40,VIX3M,VIX6M,VIX1Y
Date,,,,,,,,
1998-10-01,1017.01,4695.37,40.95,124.22,NaN,NaN,NaN,NaN
1998-11-01,1098.67,5357.66,28.05,121.06,NaN,NaN,NaN,NaN
1998-12-01,1163.63,5202.07,26.01,87.22,NaN,NaN,NaN,NaN
1999-01-01,1229.23,5025.85,24.42,121.13,NaN,NaN,NaN,NaN
1999-02-01,1279.64,5427.55,26.25,87.16,NaN,NaN,NaN,NaN


In [39]:
# Rolling realized volatility:

def realized_vol_monthly(series, window_months=1):

    log_returns = np.log(series / series.shift(1))
    return log_returns.rolling(window_months).std() * np.sqrt(12)  # annualized

# Local equity vol (JALSH)
df["LocalEquityVol1"]  = realized_vol_monthly(df["JALSH"], 1)
df["LocalEquityVol3"]  = realized_vol_monthly(df["JALSH"], 3)
df["LocalEquityVol6"]  = realized_vol_monthly(df["JALSH"], 6)
df["LocalEquityVol12"] = realized_vol_monthly(df["JALSH"], 12)

# Global equity vol (S&P500)
df["GlobalEquityVol1"]  = realized_vol_monthly(df["SPX"], 1)
df["GlobalEquityVol3"]  = realized_vol_monthly(df["SPX"], 3)
df["GlobalEquityVol6"]  = realized_vol_monthly(df["SPX"], 6)
df["GlobalEquityVol12"] = realized_vol_monthly(df["SPX"], 12)


df.head()

,SPX,JALSH,VIX,MOVE,SAVIT40,VIX3M,VIX6M,VIX1Y,LocalEquityVol1,LocalEquityVol3,LocalEquityVol6,LocalEquityVol12,GlobalEquityVol1,GlobalEquityVol3,GlobalEquityVol6,GlobalEquityVol12
Date,,,,,,,,,,,,,,,,
1998-10-01,1017.01,4695.37,40.95,124.22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1998-11-01,1098.67,5357.66,28.05,121.06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1998-12-01,1163.63,5202.07,26.01,87.22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1999-01-01,1229.23,5025.85,24.42,121.13,NaN,NaN,NaN,NaN,NaN,0.327947,NaN,NaN,NaN,0.042419,NaN,NaN
1999-02-01,1279.64,5427.55,26.25,87.16,NaN,NaN,NaN,NaN,NaN,0.217891,NaN,NaN,NaN,0.032222,NaN,NaN


In [40]:
# VIX term structure:

vix_term = df[["VIX", "VIX3M", "VIX6M", "VIX1Y"]].dropna()
pca = PCA(n_components=2).fit(vix_term)
pcs = pca.transform(vix_term)

df.loc[vix_term.index, "VolTermStructPC1"] = pcs[:, 0]
df.loc[vix_term.index, "VolTermStructPC2"] = pcs[:, 1]


In [41]:
#  Volatility spreads:

df["VolSpread1M"]   = df["VIX3M"] - df["VIX"]
df["VolSpread6M"]   = df["VIX6M"] - df["VIX"]
df["VixSaviSpread"] = df["VIX"] - df["SAVIT40"]
df["VixMoveSpread"] = df["VIX"] - df["MOVE"]



In [42]:
vol_features = df[['LocalEquityVol3', 'LocalEquityVol6',
       'LocalEquityVol12', 'GlobalEquityVol3',
       'GlobalEquityVol6', 'GlobalEquityVol12', 'VolTermStructPC1',
       'VolTermStructPC2', 'VolSpread1M', 'VolSpread6M', 'VixSaviSpread',
       'VixMoveSpread']]

vol_features.describe(include="all").T
vol_features.to_csv("volatility_features.csv", index=True)